# 4D SfM Pipeline

Processes one new day at a time through the full 4D SfM workflow. All logic
lives in `cntp.pipeline_4dsfm.run_4dsfm_day`; this notebook is just config +
one call.

Steps run by `run_4dsfm_day` (each skips if its output already exists):

1. Multi-temporal bundle adjustment — reference cameras pinned tight (0.001 m),
   new-day cameras loose; new-day IOP floating, ref IOP fixed per-day.
2. Single-day re-run with **fixed** IOP from Step 1.
3. ASP three-stage ICP (`point-to-plane` → `similarity-p2p` → stable terrain only).
4. Apply ASP transform to the single-day camera EOPs.
5. _(no-op — verification only)_
6. Rebuild the cloud from the corrected Metashape chunk transform (same session
   as the matrix fix, so it isn't lost on save/reload).
7. Append the validated day to the reference registry.

Step 3b runs M3C2 on the coreg cloud; Step 6b validates that the rebuilt cloud
matches the coreg cloud (median ≈ 0 m → transform propagated correctly).


In [12]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from cntp.pipeline_4dsfm import run_4dsfm_day


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

Set all paths and parameters here. Defaults match the values that produced the
validated 2023-12-15 run; tweak per-day if needed.


In [19]:
# ── Paths ────────────────────────────────────────────────────────────────
base_dir     = Path("/mnt/g/2023_11_Nepal/2023_Changri")
tlcam_dir    = base_dir / "TLCAM"
output_dir   = base_dir

ref_cloud    = base_dir / "Ref_PC" / "Reference_UAV_TLC_PCS.laz"
glacier_mask = base_dir / "glaciermask_new" / "glacier_mask_pcs.shp"
registry_csv = output_dir / "output_new" / "reference_registry.csv"

# ── Date to process ──────────────────────────────────────────────────────
new_date = "2024-08-17"

# ── Pipeline parameters (defaults shown — override as needed) ────────────
params = dict(
    match_downscale = 1,
    depth_downscale = 2,
    loc_acc_new     = (0.5, 0.5, 0.5),
    rot_acc_new     = (5.0, 5.0, 5.0),
    ref_downsample  = 0.4,
    tba_downsample  = 1.0,
    p2p_max_disp    = 10.0,
    sp2p_max_disp   =  5.0,
    m_sp2p_max_disp =  1,
    use_ecef        = True,
    overwrite       = False,   # True forces full recompute
    verbose         = True,    # print pc_align stdout
)


## Run

`overwrite=False` means each step skips itself if its key output exists on
disk — useful for resuming after a crash. Set `overwrite=True` above for a
fresh run.


In [21]:
result = run_4dsfm_day(
    new_date     = new_date,
    tlcam_dir    = tlcam_dir,
    ref_cloud    = ref_cloud,
    glacier_mask = glacier_mask,
    registry_csv = registry_csv,
    output_dir   = output_dir,
    stop_after_ba= False,
    **params,
)

print()
print(f"Coreg M3C2  : before {result['coreg_med_before']:+.4f} m  →  after {result['coreg_med_after']:+.4f} m")
print(f"Validation  : median {result['validation_med']:+.4f} m  std {result['validation_std']:.4f} m")
print(f"Validated   : {result['validated_laz']}")


[Step 1] Skipping — 2024-08-17_cameras_4DSfM.csv exists

[Step 2] Single-day fixed IOP — 2024-08-17

  Single-day BA (fixed IOP) : 2024-08-17   (46 images)
SaveProject: path = /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17.psx
saved project in 0.058901 sec
LoadProject: path = /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17.psx
loaded project in 0.02494 sec
AddPhotos
  Added 46 photos
  Configured 5 sensors (IOP fixed)
  Set references from CSV : 32/46 cameras matched
  EOP priors (loose 0.5 m / 5.0°) : 32/46
  Matching photos ...
MatchPhotos: downscale = 1, generic_preselection = on, reference_preselection = on, filter_mask = off, mask_tiepoints = on, filter_stationary_points = on, keypoint_limit = 80000, keypoint_limit_per_mpx = 1000, tiepoint_limit = 8000, guided_matching = off, exclude_corners = off
saved matching data in 0.011052 sec
saved object list in 0.005041 sec
scheduled 3 keypoint detection groups
saved keypoint pa

Can't load OpenCL library


[GPU] photo 0: 80000 points
[GPU] photo 3: 80000 points
[GPU] photo 6: 80000 points
[GPU] photo 9: 80000 points
[GPU] photo 12: 80000 points
[GPU] photo 15: 80000 points
[GPU] photo 18: 80000 points
[GPU] photo 21: 80000 points
[GPU] photo 24: 80000 points
[GPU] photo 27: 80000 points
[GPU] photo 30: 80000 points
[GPU] photo 33: 80000 points
[GPU] photo 36: 80000 points
[GPU] photo 39: 80000 points
[GPU] photo 42: 80000 points
[GPU] photo 45: 80000 points
points detected in 8.24499 sec
loaded object list in 0.00277 sec
loaded keypoint partition in 0.003126 sec
loaded matching data in 0.002354 sec
Found 1 GPUs in 0.000812 sec (CUDA: 2.1e-05 sec, OpenCL: 0.000386 sec, Vulkan: 0.000384 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]


Can't load OpenCL library


[GPU] photo 1: 80000 points
[GPU] photo 4: 80000 points
[GPU] photo 7: 80000 points
[GPU] photo 10: 80000 points
[GPU] photo 13: 80000 points
[GPU] photo 16: 80000 points
[GPU] photo 19: 80000 points
[GPU] photo 22: 80000 points
[GPU] photo 25: 80000 points
[GPU] photo 28: 80000 points
[GPU] photo 31: 80000 points
[GPU] photo 34: 80000 points
[GPU] photo 37: 80000 points
[GPU] photo 40: 80000 points
[GPU] photo 43: 80000 points
points detected in 7.61014 sec
loaded object list in 0.002676 sec
loaded keypoint partition in 0.003248 sec
loaded matching data in 0.002446 sec
Found 1 GPUs in 0.000609 sec (CUDA: 1.6e-05 sec, OpenCL: 0.000298 sec, Vulkan: 0.00028 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]


...

Found 1 GPUs in 0.072611 sec (CUDA: 0.00355 sec, OpenCL: 0.058184 sec, Vulkan: 0.010358 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
group 1/1: cameras images prepared in 9.37218 s
group 1/1: 46 x frame
group 1/1: 46 x uint8
group 1/1: expected peak VRAM usage: 897 MB (404 MB max alloc, 6512x7326 mipmap texture, 13 max neighbors)
Found 1 GPUs in 0.001162 sec (CUDA: 1.9e-05 sec, OpenCL: 0.00051 sec, Vulkan: 0.000592 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
Using device 'NVIDIA GeForce RTX 3050 Ti Laptop GPU' in concurrent. (2 times)
Camera 9 skipped (no neighbors)
Camera 44 skipped (no neighbors)
[GPU 1] group 1/1: estimating

Can't load OpenCL library


[GPU 1] Camera 0 samples after final filtering: 39% (2.2613 avg inliers) = 100% - 12% (not matched) - 18% (bad matched) - 3% (no neighbors) - 6% (no cost neighbors) - 12% (inconsistent normal) - 0% (estimated bad angle) - 0% (found bad angle) - 9% (speckles filtering)
[GPU 1] Camera 0 tile #1/4: level #6/6 (x2 downscale: 1664x1280, image blowup: 3328x2560) done in 1.54583 s = 42% propagation + 44% refinement + 7% filtering + 0% smoothing
Peak VRAM usage updated: Camera 0 (6 neihbs): 277 MB = 136 MB gpu_neighbImages (49%) + 30 MB gpu_mipmapNeighbImage (11%) + 24 MB gpu_tmp_hypo_ni_cost (9%) + 12 MB gpu_neighbMasks (5%) + 12 MB gpu_tmp_normal (4%) + 8 MB gpu_refImage (3%) + 8 MB gpu_depth_map (3%) + 8 MB gpu_cost_map (3%) + 8 MB gpu_coarse_depth_map_radius (3%) + 8 MB gpu_coarse_depth_map (3%)
[GPU 2] Camera 1 samples after final filtering: 42% (2.91521 avg inliers) = 100% - 21% (not matched) - 13% (bad matched) - 2% (no neighbors) - 5% (no cost neighbors) - 11% (inconsistent normal) - 0

Can't load OpenCL library


Camera 0 (6 neighbs) level #1/3 filtering: 8% good (5% of speckles) + 8% norm (95% of speckles) - 0% speckles + 5% bad + 78% empty (29% inliers support + 6% inliers intersects + 14% inliers doesn't reach + 6% inliers no depth + 34% outliers support + 11% outliers intersects + 26% outliers doesn't reach + 5% inliers occludes + 11% outliers occludes)
Camera 0 (6 neighbs) level #2/3 filtering: 13% good (3% of speckles) + 12% norm (97% of speckles) - 0% speckles + 7% bad + 68% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 87% outliers support + 13% outliers intersects + 41% outliers doesn't reach + 0% inliers occludes + 64% outliers occludes)
Camera 0 (6 neighbs) level #3/3 filtering: 16% good (3% of speckles) + 10% norm (97% of speckles) - 0% speckles + 9% bad + 64% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 91% outliers support + 9% outliers intersects + 33% outliers doesn't r

## DEM + Orthoimage

Two rasters per cloud, both anchored to the reference cloud's XY bounding
box so every output lands on the same pixel grid (directly stackable and
differenceable across days).

1. **Reference** (one-time) — DEM + ortho of the reference cloud itself,
   cached under `_ref_cache/`. Skipped on re-runs.
2. **Per day** — DEM + ortho of the day's co-registered cloud, written
   alongside the project in `<date>/single_day/`.

In [5]:
import laspy
from cntp.io import build_reference_dem_and_ortho

# ── Knobs ──────────────────────────────────────────────────────────
ref_cloud_downsample = 0.1     # extra downsample fed to griddata
res                  = 1.0      # output pixel size [m]
max_gap_pixels       = 1
overwrite_ref_dem    = True    # True to force rebuild

# ── Paths derived from config (cell 3) ────────────────────────────
ref_cache_dir = output_dir / "output_new" / "_ref_cache"
ref_ds_path   = ref_cache_dir / f"{ref_cloud.stem}_ds{params['ref_downsample']:.2f}.las"
ref_input     = ref_ds_path if ref_ds_path.exists() else ref_cloud

with laspy.open(ref_cloud) as _f:
    utm_epsg = _f.header.parse_crs().to_epsg()

# ── Build ──────────────────────────────────────────────────────────
ref_dem, ref_ortho = build_reference_dem_and_ortho(
    ref_cloud_path   = ref_input,
    cache_dir        = ref_cache_dir,
    res              = res,
    max_gap_pixels   = max_gap_pixels,
    utm_epsg         = utm_epsg,
    cloud_downsample = ref_cloud_downsample,
    overwrite        = overwrite_ref_dem,
)

print(f"UTM EPSG  : {utm_epsg}")
print(f"Ref input : {ref_input.name}")
print(f"Ref DEM   : {ref_dem}")
print(f"Ref Ortho : {ref_ortho}")

  Loaded 5,044,598 pts after XY clip (downsample=0.1, grid 1307×1248 @ 1.0 m, EPSG:32645)
Saved: /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_dem.tif
Saved orthoimage: /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_ortho.tif
UTM EPSG  : 32645
Ref input : Reference_UAV_TLC_PCS_ds0.40.las
Ref DEM   : /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_dem.tif
Ref Ortho : /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_ortho.tif


In [22]:
import laspy
from cntp.io import build_dem_and_ortho

# ── Knobs ──────────────────────────────────────────────────────────
res                  = 1.0      # output pixel size [m]
max_gap_pixels       = 1
overwrite_day_dem    = True    # True to force rebuild

# ── Paths derived from config (cell 3) ────────────────────────────
day_dir     = output_dir / "output_new" / new_date
aligned_las = day_dir / "coreg" / f"{new_date}_cloud_coreg_hsfm.las"
single_day  = day_dir / "single_day"

with laspy.open(ref_cloud) as _f:
    utm_epsg = _f.header.parse_crs().to_epsg()

# ── Build ──────────────────────────────────────────────────────────
dem, ortho = build_dem_and_ortho(
    cloud_las        = aligned_las,
    ref_las          = ref_cloud,
    out_dir          = single_day,
    name_stem        = new_date,
    res              = res,
    max_gap_pixels   = max_gap_pixels,
    utm_epsg         = utm_epsg,
    cloud_downsample = params['tba_downsample'],
    overwrite        = overwrite_day_dem,
)

print(f"Day DEM   : {dem}")
print(f"Day Ortho : {ortho}")

  Loaded 7,740,611 pts after XY clip (downsample=1.0, grid 1307×1248 @ 1.0 m, EPSG:32645)
Saved: /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17_dem.tif
Saved orthoimage: /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17_ortho.tif
Day DEM   : /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17_dem.tif
Day Ortho : /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/2024-08-17_ortho.tif


## DoD (DEM of Difference)

Pixel-wise `ref_dem - day_dem` over the common pixel grid. Output is
masked to the intersection of valid coverage (effectively the day's
footprint inside the reference's wider footprint). Sign: positive
means the reference's surface sits higher than the day.

Written as `<output_new>/<date>/single_day/DOD.tif`.

In [23]:
from cntp.io import build_dod

# ── Knobs ──────────────────────────────────────────────────────────
overwrite_dod = True    # True to force rebuild

# ── Paths derived from config (cell 3) ────────────────────────────
ref_cache_dir = output_dir / "output_new" / "_ref_cache"
single_day    = output_dir / "output_new" / new_date / "single_day"

ref_dem_path = ref_cache_dir / "reference_dem.tif"
day_dem_path = single_day   / f"{new_date}_dem.tif"

# ── Build ──────────────────────────────────────────────────────────
dod_path = build_dod(
    ref_dem_path = ref_dem_path,
    day_dem_path = day_dem_path,
    out_path     = single_day / "DOD.tif",
    overwrite    = overwrite_dod,
)

print(f"DoD : {dod_path}")

Saved: /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/DOD.tif
  DoD valid pixels : 176,682/1,631,136 (10.8%)
DoD : /mnt/g/2023_11_Nepal/2023_Changri/output_new/2024-08-17/single_day/DOD.tif


### DoD histogram

Histogram of the finite DoD pixels with median / mean / σ in the legend.
Saved alongside `DOD.tif` as `dod_histogram.png`.

In [ ]:
import rasterio
from cntp.plot import plot_dod_histogram

# ── Paths derived from config (cell 3) ────────────────────────────
single_day = output_dir / "output_new" / new_date / "single_day"
dod_path   = single_day / "DOD.tif"

with rasterio.open(dod_path) as src:
    dod_values = src.read(1)

stats = plot_dod_histogram(
    dod_values,
    output_dir = single_day,
    title      = new_date,
    filename   = "dod_histogram.png",
)
print(stats)

In [ ]:
import rasterio

for label, p in [
    ("ref_dem",  ref_cache_dir / "reference_dem.tif"),
    ("day_dem",  single_day   / f"{new_date}_dem.tif"),
    ("DOD",      single_day   / "DOD.tif"),
]:
    with rasterio.open(p) as src:
        print(f"{label:8} CRS={src.crs}  shape={src.shape}  res={src.res}")


In [8]:
import numpy as np
import rasterio

with rasterio.open(output_dir / output_new / new_date / single_day/"DOD.tif") as src:
    data = src.read(1, masked=True)
    
    print(f"Total valid pixels: {data.count()}")
    print(f"Pixels = 0:        {np.sum(data == 0)}")
    print(f"Pixels < -1000:    {np.sum(data < -1000)}")
    print(f"Pixels < -100:     {np.sum(data < -100)}")
    print(f"Pixels > 100:      {np.sum(data > 100)}")
    print(f"Real min (>-1000): {data[data > -1000].min()}")

NameError: name 'output_new' is not defined

In [24]:
from cntp.io import build_dod
from cntp.plot import plot_dod_histogram
import rasterio

# ── Configure ─────────────────────────────────────────────────────────
date1 = "2023-12-15"   # earlier epoch
date2 = "2024-04-29"   # later epoch — adjust to your real April date

# Sign convention follows build_dod(ref, day) → result = ref - day.
# Pass earlier as `ref` for a "earlier − later" DoD (positive = melt
# between the two if you flip the calendar; pick whichever direction
# matches the story you want the sign to tell).
dem_earlier = output_dir / "output_new" / date1 / "single_day" / f"{date1}_dem.tif"
dem_later   = output_dir / "output_new" / date2 / "single_day" / f"{date2}_dem.tif"

# Dedicated output dir for multi-day DoDs (separate from per-day DOD.tif).
multi_dod_dir = output_dir / "output_new" / "_dod_multiday"
multi_dod_dir.mkdir(parents=True, exist_ok=True)

dod_name = f"DOD_{date1}_minus_{date2}.tif"        # ← matches sign convention
dod_path = build_dod(
    ref_dem_path = dem_earlier,
    day_dem_path = dem_later,
    out_path     = multi_dod_dir / dod_name,
    overwrite    = False,
)

# ── Histogram ─────────────────────────────────────────────────────────
with rasterio.open(dod_path) as src:
    values = src.read(1)

stats = plot_dod_histogram(
    values,
    output_dir = multi_dod_dir,
    title      = f"{date1} − {date2}",
    filename   = f"{dod_path.stem}_histogram.png",
)
print(stats)
print("DoD :", dod_path)


  DoD cached → DOD_2023-12-15_minus_2024-04-29.tif
{'median': 0.08653540663908643, 'mean': 0.10446990850711037, 'std': 7.361202752021692, 'n': 184845}
DoD : /mnt/g/2023_11_Nepal/2023_Changri/output_new/_dod_multiday/DOD_2023-12-15_minus_2024-04-29.tif
